# Maintaining a Type 1 dimension with Delta Lake

This notebook uses Apache Spark and Delta Lake to prepare an initial Superstore dataset, persist it as Delta data, and apply an incremental Type 1 MERGE.

## Part 1: Configure Delta and start Spark

The Delta dependencies are configured before the Spark session is created.

In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession
from delta.tables import DeltaTable

# Resolve the project and dependency locations
project_dir = Path.cwd().resolve()
if not (project_dir / "data").exists() and (project_dir.parent / "data").exists():
    project_dir = project_dir.parent

delta_jar = str(project_dir / "delta-spark_2.12-3.2.0.jar")
storage_jar = str(project_dir / "delta-storage-3.2.0.jar")
delta_table_path = str(project_dir / "delta_tables" / "superstore_scd")

spark = SparkSession.builder \
    .appName("Delta_SCD_Pipeline") \
    .master("local[*]") \
    .config("spark.jars", f"{delta_jar},{storage_jar}") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark Session with Delta Lake support created successfully.")


## Part 2: Import and prepare the initial records

The original Superstore CSV is loaded and its column names are standardized.

In [ ]:
master_csv = str(project_dir / "data" / "Sample - Superstore.csv")
master_raw = spark.read.option("header", True).option("inferSchema", True).csv(master_csv)

print(f"Ingested baseline master CSV containing {master_raw.count()} records.")

# Make column names storage-safe by replacing spaces and symbols
clean_cols = [c.replace(" ", "_").replace("-", "_") for c in master_raw.columns]
master_df = master_raw.toDF(*clean_cols)

# Rename identifier fields to the chosen convention
master_df = master_df.withColumnRenamed("Row_ID", "ID") \
                     .withColumnRenamed("Order_ID", "OrderID") \
                     .withColumnRenamed("Customer_ID", "CustomerID") \
                     .withColumnRenamed("Customer_Name", "CustomerName") \
                     .withColumnRenamed("Postal_Code", "PostalCode")

master_df = master_df.dropDuplicates(["ID"])
master_df = master_df.na.fill({"City": "Unknown", "State": "Unknown"})

master_df.printSchema()


## Part 3: Write the starting Delta table

The cleaned master dataset is saved in Delta format to establish the initial table.

In [ ]:
# Persist the initial dataset to the local Delta path
master_df.write.format("delta").mode("overwrite").save(delta_table_path)
print(f"Initial baseline Delta table successfully created at: {delta_table_path}")


## Part 4: Read the incremental batch

The incoming records are loaded and adjusted to match the base table schema.

In [ ]:
incremental_csv = str(project_dir / "data" / "superstore_incremental.csv")
incremental_raw = spark.read.option("header", True).option("inferSchema", True).csv(incremental_csv)

print(f"Ingested {incremental_raw.count()} incremental update records.")

# Bring the incremental fields into line with the master fields
inc_clean_cols = [c.replace(" ", "_").replace("-", "_") for c in incremental_raw.columns]
incremental_df = incremental_raw.toDF(*inc_clean_cols)

incremental_df.show(5, truncate=False)


## Part 5: Apply the Type 1 MERGE

The Delta MERGE uses `ID` as its match key to refresh existing rows and append new ones.

In [ ]:
print("Opening delta storage folder to initiate merge...")
delta_target = DeltaTable.forPath(spark, delta_table_path)

(
    delta_target.alias("target")
    .merge(
        incremental_df.alias("source"),
        "target.ID = source.ID"
    )
    .whenMatchedUpdate(set={
        "City": "source.City",
        "State": "source.State",
        "Sales": "source.Sales",
        "Profit": "source.Profit"
    })
    .whenNotMatchedInsertAll()
    .execute()
)
print("Delta MERGE (SCD Type 1) finished successfully.")


## Part 6: Verify the merged output

The final table is reviewed and checked for repeated identifiers.

In [ ]:
# Open the Delta table after the merge
final_df = spark.read.format("delta").load(delta_table_path)

print("Preview of updated records:")
final_df.orderBy("ID") \
        .select("ID", "CustomerName", "City", "State", "Sales", "Profit") \
        .show(15, truncate=False)

# Perform the final data-quality checks
total_rows = final_df.count()
duplicate_count = final_df.groupBy("ID").count().filter("count > 1").count()

print(f"Validation Results:")
print(f"  - Total records post-merge: {total_rows}")
print(f"  - Duplicate ID occurrences:  {duplicate_count}")

# End the Spark session
spark.stop()
